In [20]:
import os
import sys
module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

print(module_path)

import numpy as np
import torch
import torch.nn as nn
from hedging.envs import HedgeCallBS
from hedging.logit_normal import LogitNormal
from hedging.plot_utils import plot_portfolio_vs_option_price

from torchrl.envs import GymWrapper
from torchrl.envs.utils import ExplorationType, set_exploration_type
from torchrl.modules import ProbabilisticActor, SafeModule, NormalParamExtractor, SafeProbabilisticModule

import torch.nn as nn
from tensordict.nn import TensorDictModule, TensorDictSequential



/Users/manu13/Desktop/PHD/DeepHedging/deep_hedging_v0


In [21]:
# --- Env Parameters ---
S0 = np.array([50.0, 100.0, 200.0])
K  = np.array([[45.0, 55.0], [90.0, 110.0], [180.0, 220.0]])
maturity = 1.0
r = 0.03
sigma = np.array([0.15, 0.2, 0.25])
num_paths = 100
num_steps = 250
history_len = 5
input_dim = 11
hidden_size = 64
action_dim = 1

base_env = HedgeCallBS(S0, K, maturity, r, sigma, num_paths, num_steps, history_len=history_len)
env = GymWrapper(base_env)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
act_spec = env.specs["input_spec", "full_action_spec", "action"].to(device)


In [22]:
frames_per_batch = env.num_envs * num_steps
sub_batch_num = 10
sub_batch_size = frames_per_batch // sub_batch_num
frames_per_batch, sub_batch_size

(150000, 15000)

In [23]:
class FeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.LSTM(
            input_size=11, hidden_size=64, num_layers=2, batch_first=True, dropout=0.0
        )

    def forward(self, x):
        if len(x.shape) > 3:  # Handle 4D input
            x_reshaped = x.view(-1, x.shape[-2], x.shape[-1])
            output, _ = self.rnn(x_reshaped)
            output = output.view(
                x.shape[0], x.shape[1], x.shape[2], output.shape[-1]
            )
        else:
            output, _ = self.rnn(x)
        output = output[..., -1, :]  # last time step
        return output

# Extract features
feature_extractor = SafeModule(
    module=FeatureExtractor(),
    in_keys=["observation"],
    out_keys=["feature"],
)

# Deterministic mapping -> distribution params
policy_network = TensorDictModule(
    nn.Sequential(
        nn.Linear(64, 2 * action_dim),
        NormalParamExtractor(),   # splits into loc, scale
    ),
    in_keys=["feature"],
    out_keys=["loc", "scale"],
)

# Wrap with SafeProbabilisticModule
actor = SafeProbabilisticModule(
    in_keys=["loc", "scale"],     # distribution parameters
    out_keys=["action"],          # sampled action
    distribution_class=LogitNormal,
    return_log_prob=True,
)

actor = TensorDictSequential(feature_extractor, policy_network, actor).to(device)


In [24]:
num_epochs = 20
num_episodes = 200
gamma = 0.999
learning_rate = 1e-5
optimizer = torch.optim.Adam(actor.parameters(), lr=learning_rate)

In [25]:
def discounted_returns(rewards: torch.Tensor, gamma: float) -> torch.Tensor:
    # rewards: [T, B]
    T, B = rewards.shape
    idx = torch.arange(T, device=rewards.device)
    exp_mat = idx[None, :] - idx[:, None]             # [T, T]
    disc_mat = torch.triu(torch.pow(
        torch.as_tensor(gamma, device=rewards.device), exp_mat).to(rewards.dtype)
    )
    return disc_mat @ rewards                          # [T, B]

In [26]:
with set_exploration_type(ExplorationType.RANDOM):
    for epoch in range(num_epochs):
        for episode in range(num_episodes):
            # mirror baseline seeding cadence
            env.set_seed(epoch + 1000)

            actor.train()
            td = env.rollout(
                policy=actor,
                max_steps=int(num_steps * maturity),
                auto_reset=True,
                auto_cast_to_device=True,
                break_when_all_done=True,   # full ep for all envs
            )

            # shapes: [T, B]
            rewards = td["next", "reward"].squeeze(-1)
            log_probs = td["action_log_prob"].squeeze(-1)

            # discounted returns and normalization across envs (axis=1)
            R = discounted_returns(rewards, gamma)     # [T, B]
            eps = torch.finfo(R.dtype).eps
            R = R - R.mean(dim=1, keepdim=True)
            R = R / (R.std(dim=1, keepdim=True) + eps)

            # reinforce loss
            loss = (-(R * log_probs)).mean()

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(actor.parameters(), max_norm=1.0)
            # nansafe (shouldn’t trigger, but keeps parity with your PPO hygiene)
            for p in actor.parameters():
                if p.grad is not None:
                    p.grad = torch.nan_to_num(p.grad)
            optimizer.step()

            if (episode + 1) % 10 == 0:
                avg_reward = rewards.mean().item()     # per-step mean over T×B
                print(
                    f"Epoch {epoch+1}/{num_epochs}, Episode {episode + 1}/{num_episodes}, "
                    f"Loss: {loss.item():.6f}, Avg. Reward: {avg_reward:.6f}"
                )

Epoch 1/20, Episode 10/200, Loss: -0.000960, Avg. Reward: -4.743216
Epoch 1/20, Episode 20/200, Loss: -0.000677, Avg. Reward: -4.763454
Epoch 1/20, Episode 30/200, Loss: -0.001863, Avg. Reward: -4.733386
Epoch 1/20, Episode 40/200, Loss: -0.000870, Avg. Reward: -4.775404
Epoch 1/20, Episode 50/200, Loss: 0.000179, Avg. Reward: -4.686666
Epoch 1/20, Episode 60/200, Loss: 0.001415, Avg. Reward: -4.764295


KeyboardInterrupt: 

In [27]:
base_env = HedgeCallBS(S0, K, maturity, r, sigma, 5, num_steps, history_len=history_len)
env = GymWrapper(base_env, device=device)
env.reset(seed=0)

with set_exploration_type(ExplorationType.DETERMINISTIC):
    rollout = env.rollout(max_steps=num_steps, policy=actor)


In [28]:
rewards = rollout['next', 'reward'].detach().cpu().numpy()
rewards.min(), rewards.max(), rewards.mean(), rewards.std()

(-69.54177, -0.00029242566, -4.028334, 5.797201)

In [29]:
plot_portfolio_vs_option_price(env._env)